In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MobileMoneyAnalytics").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/abijedidiah/spark_easter_2026/venv/lib64/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/18 09:22:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.csv("synthetic_mobile_money_transaction_dataset.csv", header=True, inferSchema=True)

In [3]:
from pyspark.sql.functions import col

result = df.filter(col('amount') > 1000)
print('Instant. A plan was recorded, no data was read.')

Instant. A plan was recorded, no data was read.


In [4]:
big = df.filter(col("amount") > 1000)
big_small = big.select("step", "amount", "isFraud")
print("Two orders stacked. Still nothing executed.")

Two orders stacked. Still nothing executed.


In [5]:
result = df.filter(col("amount") > 1000)
result.count()

1150064

In [6]:
result.explain("formatted")

== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [10]: [step#17, transactionType#18, amount#19, initiator#20L, oldBalInitiator#21, newBalInitiator#22, recipient#23, oldBalRecipient#24, newBalRecipient#25, isFraud#26]
Batched: false
Location: InMemoryFileIndex [file:/home/abijedidiah/spark_easter2026/synthetic_mobile_money_transaction_dataset.csv]
PushedFilters: [IsNotNull(amount), GreaterThan(amount,1000.0)]
ReadSchema: struct<step:int,transactionType:string,amount:double,initiator:bigint,oldBalInitiator:double,newBalInitiator:double,recipient:string,oldBalRecipient:double,newBalRecipient:double,isFraud:int>

(2) Filter [codegen id : 1]
Input [10]: [step#17, transactionType#18, amount#19, initiator#20L, oldBalInitiator#21, newBalInitiator#22, recipient#23, oldBalRecipient#24, newBalRecipient#25, isFraud#26]
Condition : (isnotnull(amount#19) AND (amount#19 > 1000.0))




In [7]:
%%time
rows = df.select("amouunt").limit(50000).collect()
count = 0
for row in rows:
    if row.amount > 1000:
        count += 1
print("loop count (first 50k rows only!)", count)

{"ts": "2026-10-18 09:23:09.671", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amouunt` cannot be resolved. Did you mean one of the following? [`amount`, `step`, `isFraud`, `recipient`, `initiator`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "104)", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o33.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amouunt` cannot be resolved. Did you mean one of the following? [`amount`, `step`, `isFraud`, `recipient`, `initiator`]. SQLSTATE: 42703;\n'Project ['amouunt]\n+- Relation [step#17,transactionType#18,amount#19,initiator#20L,oldBalInitiator#

CPU times: user 17.3 ms, sys: 7.42 ms, total: 24.7 ms
Wall time: 318 ms


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `amouunt` cannot be resolved. Did you mean one of the following? [`amount`, `step`, `isFraud`, `recipient`, `initiator`]. SQLSTATE: 42703;
'Project ['amouunt]
+- Relation [step#17,transactionType#18,amount#19,initiator#20L,oldBalInitiator#21,newBalInitiator#22,recipient#23,oldBalRecipient#24,newBalRecipient#25,isFraud#26] csv


In [8]:
%%time
rows = df.select("amount").limit(50000).collect()
count = 0
for row in rows:
    if row.amount > 1000:
        count += 1
print("loop count (first 50k rows only!)", count)

loop count (first 50k rows only!) 22220
CPU times: user 243 ms, sys: 27.4 ms, total: 270 ms
Wall time: 638 ms


In [9]:
%%time
print("engine count (all rows):", df.filter(col("amount") > 1000).count())

engine count (all rows): 1150064
CPU times: user 3.55 ms, sys: 1.83 ms, total: 5.38 ms
Wall time: 649 ms


------------------------—-------------—--------—------------—-------------—-----------—---------------—--------

In [10]:
import time           
import pandas as pd    

t = time.time()                                                   
pdf = pd.read_csv("synthetic_mobile_money_transaction_dataset.csv") 
print(f"pandas load: {time.time()-t:.2f}s  <- data IS in memory now")

pandas load: 2.66s  <- data IS in memory now


In [14]:
t = time.time()
sdf = spark.read.csv("synthetic_mobile_money_transaction_dataset.csv", header=True, inferSchema=True)
print(f"spark read: {time.time()-t:.2f}s  <- just a plan")
sdf.printSchema() 

[Stage 16:===>                                                    (1 + 15) / 16]

spark read: 1.26s  <- just a plan
root
 |-- step: integer (nullable = true)
 |-- transactionType: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- initiator: long (nullable = true)
 |-- oldBalInitiator: double (nullable = true)
 |-- newBalInitiator: double (nullable = true)
 |-- recipient: string (nullable = true)
 |-- oldBalRecipient: double (nullable = true)
 |-- newBalRecipient: double (nullable = true)
 |-- isFraud: integer (nullable = true)



In [12]:
df.select("step", "amount", "isFraud").limit(1000).write.mode("overwrite").parquet("mm_sample.parquet")

In [13]:
spark.read.parquet("mm_sample.parquet").show(5)

+----+--------+-------+
|step|  amount|isFraud|
+----+--------+-------+
|   0|19824.96|      1|
|   0|  598.97|      0|
|   0|  545.85|      0|
|   0|19847.01|      1|
|   0|  546.89|      0|
+----+--------+-------+
only showing top 5 rows
